# Face Emotion v3: entrenamiento con transfer learning en Colab (GPU)

Entrena `modelFEC_v3.h5` (EfficientNetB0 preentrenada en ImageNet + fine-tuning) con la GPU de Colab,
usando `train_transfer.py` y el dataset `images.rar` del repositorio.

**Pasos:**
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** → Guardar
2. **Entorno de ejecución → Ejecutar todas**
3. Al final se descarga **`modelFEC_v3.h5`**. Cópialo a la carpeta del proyecto y la app lo usa automáticamente.

Tiempo aproximado: **30-40 minutos**. No cierres la pestaña mientras entrena.

## 1. Verificar GPU e instalar Keras 2 (`tf_keras`)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ SIN GPU: activa Entorno de ejecución → Cambiar tipo de entorno → GPU"
# tf_keras con la misma versión menor que el TensorFlow de Colab (para guardar en .h5 compatible con la app)
!TFV=$(python -c "import tensorflow as tf; print('.'.join(tf.__version__.split('.')[:2]))"); pip install -q "tf_keras==${TFV}.*"

## 2. Descargar el repositorio y descomprimir el dataset

In [ ]:
%cd /content
!rm -rf Face_Emotion
!git clone --depth 1 https://github.com/RicardoGarfias/Face_Emotion.git
%cd /content/Face_Emotion
!unrar x -Y -inul images.rar ./
!for s in train validation; do echo "$s: $(find images/$s -type f | wc -l) imágenes"; done

## 3. Entrenar
- Fase 1: 3 épocas con la base congelada (solo se entrena la cabeza).
- Fase 2: hasta 20 épocas de fine-tuning de toda la red; se detiene antes si deja de mejorar.

Al final imprime la comparación con el modelo original (`modelFEC.h5`).

In [ ]:
%cd /content/Face_Emotion
!python -u train_transfer.py --img-size 96 --head-epochs 3 --ft-epochs 20 2>&1 | grep -vE "WARNING|UserWarning|saving_api|oneDNN|absl|cpu_feature"

## 4. Curvas de entrenamiento

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('/content/Face_Emotion/train_transfer_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log.epoch + 1, log.accuracy, label='entrenamiento')
ax[0].plot(log.epoch + 1, log.val_accuracy, label='validación')
ax[0].axhline(0.6486, color='gray', ls='--', label='modelo original (64.9 %)')
ax[0].axvline(3.5, color='orange', ls=':', label='inicio fine-tuning')
ax[0].set_title('Precisión'); ax[0].set_xlabel('época'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(log.epoch + 1, log.loss, label='entrenamiento')
ax[1].plot(log.epoch + 1, log.val_loss, label='validación')
ax[1].set_title('Pérdida'); ax[1].set_xlabel('época'); ax[1].legend(); ax[1].grid(True)
plt.show()
print(f"Mejor precisión en validación (sin TTA): {log.val_accuracy.max()*100:.2f} %")

## 5. Descargar el modelo
Cópialo a la carpeta del proyecto (reemplaza el `modelFEC_v3.h5` actual). La app lo usa automáticamente.

In [ ]:
from google.colab import files
files.download('/content/Face_Emotion/modelFEC_v3.h5')